# DADA2: 04 — Evaluation

Three TODOs.

You will conclude the assignment by running a RAG eval.

Three questions to answer with numbers:
- Is the decision model as good a router as the chat model? 
- Does the diagnosis agent beat the ladder, and at what cost?
- which parts of what you built were worth building?

In [1]:
from dotenv import load_dotenv
load_dotenv()

import time
from rag import config, decide, evaluate, retrieval, agent, ladder

## Read this first

This notebook is where the Gemini quota goes. The ladder agent spends about
five chat calls per question and yours about three, and the judge spends one
more on each answer. Fourteen questions through both agents is roughly 150
calls against a limit of 500 a day.

So, you will want to run it once, when notebooks 02 and 03 already pass their final cells. The comparison cells sleep between questions on purpose. Start them and leave them alone.

In [2]:
specific = evaluate.load_evalset("specific")
comparative = evaluate.load_evalset("comparative")
items = specific + comparative

print(f"{len(specific)} specific + {len(comparative)} comparative = {len(items)} labeled questions")
print("reranker:       cohere")
print("chat model:    ", config.CHAT_MODEL)
print("decision model:", config.DECISION_MODEL)

14 specific + 10 comparative = 24 labeled questions
reranker:       cohere
chat model:     gemini-3.5-flash-lite
decision model: clef-flash


## 1. Two routers

### TODO 4.1 — the decision model against the chat model (1 pt)

Every labeled question has a kind: the 14 in `evalset.json` are specific,
the 10 in `evalset_hard.json` are comparative. Ask both models which kind each
one is, and keep the time.

Write the chat-model version yourself. It is a `with_structured_output` call
with a `Literal` field, the same thing you did for `Filter` in notebook 02.
Use `decide.SCOPES` for the descriptions so both models are asked the same
question.

**Steps:**

1. Define a scoper that uses Scope to get structured output from chat model
3. In the loop below, the decision-model line is written. Add the same two lines for the chat model, keyed `"chat"`: time the call, add the seconds, add one to `right` when the choice matches `kind`.


In [3]:
from typing import Literal
from pydantic import BaseModel, Field
from rag.util import retry

# TODO 4.1 — a scoper that uses Scope to get structured output from chat model
class Scope(BaseModel):
    """Which kind of question this is."""
    scope: Literal["specific", "comparative", "figure"] = Field(
        description="; ".join(f"{k}: {v}" for k, v in decide.SCOPES.items()))

def chat_scope(question):
    t = time.time()
    out = retry(scoper.invoke, "Question about SEC 10-K filings: " + question
                + "\nWhat does it ask for?")
    return out.scope, time.time() - t

scoper = config.chat().with_structured_output(Scope)
for q in items[:5]:
    scope, t = chat_scope(q["question"])
    print(f"{scope:12} {t:.1f}s {q['question']}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


specific     4.2s What does MU state that the preparation of financial statements and related disclosures in conformity with U.S. GAAP requires management to do in its FY2024 10-K?
specific     3.9s What termination fee is AMD required to pay ZT Systems under certain circumstances related to the failure to obtain required regulatory approvals according to the FY2024 10-K?
specific     3.1s What factors does NVIDIA believe are the principal competitive factors in its market for fiscal year 2025?
specific     0.8s What competitive pressures from Intel and Nvidia did AMD face in fiscal year 2025 that could affect its market share and margins?
specific     2.2s What forms the basis for AMD's estimates and judgments used in preparing its Consolidated Financial Statements for fiscal year 2025?


In [4]:
labeled = [(i["question"], "specific") for i in specific] + \
           [(i["question"], "comparative") for i in comparative]

right = {"decision": 0, "chat": 0}
secs = {"decision": 0.0, "chat": 0.0}
for q, kind in labeled:
    t = time.time(); choice, _ = decide.scope(q); secs["decision"] += time.time() - t
    right["decision"] += (choice == kind)
    t = time.time(); choice, _ = chat_scope(q); secs["chat"] += time.time() - t
    right["chat"] += (choice == kind)
    time.sleep(3)

n = len(labeled)
for name in ("decision", "chat"):
    print(f"{name:9} {right[name]}/{n} right, {secs[name]/n:.2f}s per call")

decision  20/24 right, 0.31s per call
chat      19/24 right, 0.69s per call


NOTE: Accuracy is close, and the chat model is probably not
slower. If you expected the decision model to win on both, write down what it does give you instead: a probability per option, and a call that does not count against the daily limit. Whether those are worth having is the question
the next section answers.

## 2. Two agents

### TODO 4.2 — the ladder against the diagnosis (1 pt)

`evaluate.compare_agents` runs each agent over the questions, has the chat
model judge every answer, and keeps the bill: chat calls, decision calls, and
seconds per question. Both agents expose `run(question)` returning the final
state, which is all it needs.

Fourteen questions is enough. Use eight specific and six comparative so both
kinds are in the mix.

**Steps:** nothing to write. Run the cell once, when both agents pass their final cells. If a 429 cuts it off, score the two agents in separate cells rather than restarting the pair.

This will take a while. Plan accordingly. 

Make sure your rate limit is reset or use a different provider (e.g., Groq, OpenRouter) instead of the Gemini API. 

Let it run and come back to see your results.


In [5]:
# TODO 4.2
subset = specific[:8] + comparative[:6]

rows = evaluate.compare_agents({"ladder": ladder.run, "diagnosis": agent.run}, subset)

  agent         accuracy  chat/q  decision/q    s/q
  --------------------------------------------------


/Users/kurshatmuhammet/agentic-ai-course/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ladder            0.86     4.1         0.0   16.8
  diagnosis         0.86     2.3         2.1    8.8


In [6]:
# the judge's reasons, on anything either agent got wrong
_, notes = evaluate.score_agent(agent.run, subset[:4])
for note in notes:
    mark = "ok " if note["correct"] else "WRONG"
    print(f"[{mark}] {note['question'][:66]}  ({note['chat_calls']} chat, {note['decision_calls']} decision)")
    print(f"        {note['reason'][:150]}\n")

[ok ] What does MU state that the preparation of financial statements an  (2 chat, 2 decision)
        CORRECT
The answer accurately identifies management's responsibility to make estimates and judgments as stated in the context of financial statement p

[ok ] What termination fee is AMD required to pay ZT Systems under certa  (2 chat, 2 decision)
        CORRECT
The answer correctly identifies the $300 million termination fee related to regulatory approvals for the ZT Systems acquisition as supported b

[ok ] What factors does NVIDIA believe are the principal competitive fac  (2 chat, 2 decision)
        CORRECT
The provided answer accurately lists NVIDIA's principal competitive factors for fiscal year 2025 as stated in the filing excerpt.

[WRONG] What competitive pressures from Intel and Nvidia did AMD face in f  (2 chat, 2 decision)
        INCORRECT
The answer relies on a fabricated future fiscal year (2025) and hypothetical future events (such as a September 2025 Nvidia-Intel par

If the diagnosis agent got something wrong that the ladder got right, find
out which half failed. Run `decide.scope` and `decide.diagnose` on that
question by hand and look at the probabilities. A wrong route and a wrong
verdict are different bugs, and the fix for one is a threshold while the fix
for the other is a search strategy.

### TODO 4.3 — write it up (2 pts)

In the cell below, in your own words. Four or five short paragraphs.

**Conditions.** Reranker, chat model, decision model, k, chunk size. Enough
for someone to re-run you.

**The routers.** Which was more accurate, which was faster, and by how much.
Then say what the decision model is for, given those numbers.

**The agents.** Accuracy, chat calls per question, seconds per question, for
both. Name the question where they differed most and say why.

**One thing that did not help.** Something you built that the numbers say you
could remove. Say what the mechanism is, not that the model is better at it.

**What you would ship**, and what you would want to know about your users'
questions first.

Two things that cost marks: an improvement the table does not show, and a
difference explained by "the model is better" when the reason is in your
code.

**Steps:** write it as prose, four or five short paragraphs. The bold headings in the cell below are the order; replace the cell's text under each. Every number you quote should be in a table above.


---

### Your write-up

**Conditions.**

Cohere rerank-v3.5, chat model gemini-3.5-flash-lite, decision model clef-flash, Cloudflare qwen3-embedding-0.6b. Chunks are 1,200 characters with 150 overlap (2,124 chunks), K = 5, pool = 10, RRF constant 60; router falls back to filter+rrf below 0.6; routers tested on 24 questions and agents were tested on 14 (8 were specific 6 were comparative). Answers were judged by Gemini. In a re-run of 4 agent questions, the judge marked a diagnosis-agent answer as WRONG for citing "fiscal 2025", but the corpus includes AMD's 2025 filings. Since that was one of the 14 questions the agent scores could have the same judging error.

**The routers.**

The decision model got 20/24 right at 0.31s a call, gemini with structured output got 19/24 at 0.69 sec, so decision model is one question more accurate and around twice the speed. Real value is the probability it returns, allowing the router to be at a safe default when faced with uncertainty.

**The agents.**

Both scored 0.86 (12/14); the ladder averaged 4.1 chat calls and 16.8s per question; diagnosis agent averaged 2.3 chat calls, 2.1 decision calls and 8.8s. Notebook 03 (cells 27 and 29) shows where the savings come from, using a question outside the 14-question set: "AMD's employee dress code," which the filings don't answer. The ladder climbed all 3 rungs and spent 9 chat calls, diagnose returned absent with .92 probability which is above the agent's ABSENT_SURE cutoff of 0.8 - meaning agent answered practically instantly with 1 chat call.

**What did not help.**

RRF and reranking on top of the filter. On the 6-question sample, MRR went from 0.917 (filter, cell 27) to 0.833 (+RRF, cell 27) to 0.806 (+rerank, cell 34), and recall@5 stayed at 1.00. The drops match up with a correct passage moving down a rank on one question. Once the filter narrows the search to the right company and year, a passage from the right file is already near the top. Fusion mixes rewritten queries that rank it lower, and the reranker can only reorder those 10 passages. Without the filter, reranking helped: baseline MRR rose from 0.325 to 0.583 (cell 37). I would remove reranking from the specific-question path only because it only hurts when retrieval is already precise. Filter+rrf+rerank stays in the agent's STRONGER path. RRF stays because it's the DEFAULT and the first STRONGER step.

**What I would ship.**

The diagnosis agent since it gets identical accuracy with 44% fewer chat calls and half the latency. And START_WITH already sends specific questions to filter alone and comparative ones to hybrid+rrf. The hybrid+rrf choice is a design decision, not a measured result, because I only scored baseline and filter on the comparative set (cell 49). I would want to know how many of the user questions are comparative or unanswerable and whether users name the company and year before shipping, however.

**Copy your write-up to README.md.**